# Notebook 40 — Defence cost: what the fixed recipes cost against the default recipe

Notebooks 34-39b established that per-layer uniform pruning starves the CNN's 192-weight first layer and that
two recipe changes remove the collapse: protecting the first layer, or global magnitude pruning at the same
total sparsity. This notebook measures what those fixes cost on the deployment axes a practitioner sees:
serialized bytes, an index-value sparse payload estimate, dense CPU latency (median and p95) and throughput
at batch sizes 1 / 32 / 256 / 1024, and macro-F1, for the dense baseline and the three recipes across the
five paired seeds. Benchmark helpers and settings are those of Notebook 13 (single CPU thread, 25 warm-up
runs, 100 timed repeats) so the numbers are comparable to the archived deployment table.

Dense PyTorch timing is reported exactly as measured; unstructured zeros are not assumed to accelerate
inference. CPU runtime; no training.

In [ ]:
# --- Colab bootstrap (CPU runtime) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch
from src.config import CFG, PATHS
from src.data import load_raw, clean, temporal_within_capture_split
from src.train import load_anchor, feature_columns
from src import models as M
from src.comnet_audit import benchmark_cpu_model, serialized_state_dict_bytes, index_value_sparse_payload_bytes, environment_record, write_json

DATASET, ARCH, ARCH_KW = 'ciciot2023', 'cnn1d', {'channels': (64, 128)}
SEEDS = list(CFG['seeds']); OUT = PATHS.tables('comnet')
RECIPES = {'dense': 'M0_paired', 'default_layerwise80': 'prune80_paired', 'protect_conv0': 'layerwise80_protect_conv0_paired', 'global80': 'global80_paired'}
BATCH_SIZES = (1, 32, 256, 1024); WARMUP = 25; REPEATS = 100; TORCH_CPU_THREADS = 1
torch.set_num_threads(TORCH_CPU_THREADS)
print('recipes:', list(RECIPES), '| cpu threads:', torch.get_num_threads())

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=int(CFG['anchor_seed'])), DATASET)
feat_cols = feature_columns(df); IN_DIM = len(feat_cols)
print('features:', IN_DIM)

In [ ]:
# Sizes and CPU benchmark for every (recipe, seed)
def load_model(cell, seed):
    if cell == 'M0_paired':
        m, _, _, _ = load_anchor(DATASET, ARCH, cell, seed, arch_kwargs=ARCH_KW); return m.cpu().eval()
    m = M.build(ARCH, IN_DIM, int(df.label.nunique()), **ARCH_KW)
    ck = torch.load(PATHS.model(DATASET, ARCH, cell, seed), map_location='cpu', weights_only=False)
    m.load_state_dict(ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck); return m.eval()

size_rows, bench_rows = [], []
for recipe, cell in RECIPES.items():
    for seed in SEEDS:
        m = load_model(cell, seed)
        n = sum(p.numel() for p in m.parameters()); nz = sum(int((p != 0).sum()) for p in m.parameters())
        size_rows.append({'recipe': recipe, 'seed': seed, 'parameters': n, 'nonzero_parameters': nz, 'whole_model_sparsity': 1 - nz / n,
                          'torch_state_dict_bytes': serialized_state_dict_bytes(m), 'index_value_sparse_payload_estimate_bytes': index_value_sparse_payload_bytes(m)})
        tab = benchmark_cpu_model(m, IN_DIM, batch_sizes=BATCH_SIZES, warmup=WARMUP, repeats=REPEATS, dtype='float32')
        tab.insert(0, 'seed', seed); tab.insert(0, 'recipe', recipe); bench_rows.append(tab)
    print(f'{recipe}: benchmarked {len(SEEDS)} seeds')
sizes = pd.DataFrame(size_rows); bench = pd.concat(bench_rows, ignore_index=True)
sizes.to_csv(OUT / 'defence_cost_sizes.csv', index=False); bench.to_csv(OUT / 'defence_cost_cpu_benchmark.csv', index=False)

In [ ]:
# Summary table: cost axes + macro-F1 (from the committed Notebook 34 results) per recipe
f1 = pd.read_csv(OUT / 'cnn_policy_macro_f1_wide.csv')
f1_map = {'dense': 'M0', 'default_layerwise80': 'layerwise80_all', 'protect_conv0': 'layerwise80_protect_conv0', 'global80': 'global80'}
lat_cols = [c for c in bench.columns if 'latency' in c.lower() or 'throughput' in c.lower()]
rows = []
for recipe in RECIPES:
    s = sizes[sizes.recipe == recipe]; b1 = bench[(bench.recipe == recipe) & (bench.batch_size == 1)]; b1k = bench[(bench.recipe == recipe) & (bench.batch_size == 1024)]
    g = f1[f1.cell == f1_map[recipe]].test_macro_f1
    row = {'recipe': recipe, 'macro_f1_mean': g.mean(), 'macro_f1_sd': g.std(), 'whole_model_sparsity': s.whole_model_sparsity.mean(),
           'state_dict_bytes': s.torch_state_dict_bytes.mean(), 'sparse_payload_bytes': s.index_value_sparse_payload_estimate_bytes.mean()}
    for name, sub in (('b1', b1), ('b1024', b1k)):
        for c in lat_cols: row[f'{name}_{c}'] = float(sub[c].mean())
    rows.append(row)
summ = pd.DataFrame(rows); summ.to_csv(OUT / 'defence_cost_summary.csv', index=False)
pd.set_option('display.width', 250); print(summ.round(4).to_string(index=False))
d = summ.set_index('recipe')
print()
for r in ('protect_conv0', 'global80'):
    print(f"{r} vs default: macro-F1 {d.loc[r,'macro_f1_mean']-d.loc['default_layerwise80','macro_f1_mean']:+.3f} | sparsity {d.loc[r,'whole_model_sparsity']-d.loc['default_layerwise80','whole_model_sparsity']:+.4f} | "
          f"bytes {d.loc[r,'state_dict_bytes']/d.loc['default_layerwise80','state_dict_bytes']:.3f}x | sparse payload {d.loc[r,'sparse_payload_bytes']/d.loc['default_layerwise80','sparse_payload_bytes']:.3f}x")
write_json(OUT / 'defence_cost_environment.json', {'recipes': RECIPES, 'batch_sizes': list(BATCH_SIZES), 'warmup': WARMUP, 'repeats': REPEATS,
                                                  'torch_cpu_threads': TORCH_CPU_THREADS, 'environment': environment_record()})
print('\nDense PyTorch timing reported as measured; unstructured zeros are not assumed to accelerate inference.')

In [ ]:
# --- Commit + push: main only, own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/40_defence_cost_benchmark.ipynb'
if os.path.exists(_own):
    d_ = _json.load(open(_own))
    for c in d_.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d_, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/defence_cost_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 40: defence cost - bytes, sparse payload, CPU latency/throughput and macro-F1 for dense, default, first-layer-protected and global recipes'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)